In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_fpl_analytics AS
SELECT 
    p.player_id,
    p.first_name,
    p.last_name,
    p.known_name,
    t.team_name,
    pos.position_name,
    p.current_price_millions,
    p.total_points,
    p.minutes_played,
    p.goals_scored,
    p.assists,
    p.injury_status,
    ROUND(p.total_points / NULLIF(p.current_price_millions, 0), 2) AS points_per_million
FROM workspace.default.silver_fpl_players p
LEFT JOIN workspace.default.silver_fpl_teams t ON p.team_id = t.team_id
LEFT JOIN workspace.default.silver_fpl_positions pos ON p.position_id = pos.position_id;

In [0]:
#Vamos a leer esta tabla
df_gold = spark.table("workspace.default.gold_fpl_analytics")

#Una subcarpeta en nuestro volumen para ordenarlo un poco
export_path = "/Volumes/workspace/default/fpl_landing/export/"

#Reescribimos como un único archivo CSV (coalesce(1) para que Spark no lo divida en particiones)
(df_gold.coalesce(1).write
    .format("csv")
    .option("header", "true")
    .mode("overwrite")
    .save(export_path)
)

print("Exportación a CSV completada.")

Vamos a realizar un par de consultas para comprobar la integridad de nuestra tabla.

In [0]:
%sql
SELECT 
    COUNT(*) AS total_jugadores,
    SUM(CASE WHEN team_name IS NULL THEN 1 ELSE 0 END) AS nulos_equipo,
    SUM(CASE WHEN position_name IS NULL THEN 1 ELSE 0 END) AS nulos_posicion,
    SUM(CASE WHEN current_price_millions IS NULL THEN 1 ELSE 0 END) AS nulos_precios
FROM workspace.default.gold_fpl_analytics;

In [0]:
%sql
SELECT 
    known_name, 
    team_name, 
    position_name, 
    current_price_millions, 
    total_points, 
    points_per_million
FROM workspace.default.gold_fpl_analytics
WHERE minutes_played > 900 -- Mínimo 10 partidos completos
ORDER BY points_per_million DESC
LIMIT 10;

In [0]:
%sql
SELECT 
    team_name,
    COUNT(player_id) AS tamano_plantilla,
    SUM(goals_scored) AS goles_totales_equipo,
    ROUND(AVG(total_points), 1) AS media_puntos_por_jugador
FROM workspace.default.gold_fpl_analytics
GROUP BY team_name
ORDER BY goles_totales_equipo DESC;